# 05 Explainability

This notebook generates clear, plain-language explanations for the prioritized benefit cases identified in **Stage 6 — ML / Ranking**.

### Objectives:
1. Load the prioritized case list and model scoring details.
2. Implement a rule-based explainability engine that links model features back to human-readable reasons.
3. Generate explanations for the Top 20 cases and save the summary to `data/processed/top20_explanations.csv`.
4. Validate explanation correctness (ensuring no demographics are used, no empty explanations, and that all reasons are supported by actual data).

---

## 1. Import Required Libraries

We import standard utilities and add the project root to python directories to import our custom explainability module.

In [1]:
import pandas as pd
import numpy as np
import sys
import os

# Set printing options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 1000)

sys.path.append(os.path.abspath('..'))
from src.explainability import generate_case_explanations
print('Libraries and custom modules loaded successfully!')

Libraries and custom modules loaded successfully!


## 2. Load Scored Cases

We read the prioritized scored case dataset `scored_cases.csv` produced during model training.

In [2]:
scored_path = '../data/processed/scored_cases.csv'
df_scored = pd.read_csv(scored_path)
print(f'Scored cases dataset: {len(df_scored)} cases loaded.')

Scored cases dataset: 4200 cases loaded.


## 3. Generate Explanations

We run our explainability module pipeline on the scored cases to rank them and generate feature explanations.

In [3]:
df_explanations = generate_case_explanations(df_scored)
print(f'Explanations successfully generated for all {len(df_explanations)} cases!')

Explanations successfully generated for all 4200 cases!


## 4. Top 20 Case Explanations

We view the top 20 cases, displaying their rank, priority score, status, and the plain-language explanation.

In [4]:
df_top20 = df_explanations.head(20).copy()
display(df_top20[[
    'rank', 'case_id', 'priority_score', 'status', 
    'top_contributing_signals', 'plain_language_explanation'
]])

,rank,case_id,priority_score,status,top_contributing_signals,plain_language_explanation
0,1,C-33263,1.000000,Suspended,high_award_excess | high_contact_attempts,"Actual payment amounts were substantially higher than the standard monthly award (maximum payment was 1.99x the award, resulting in $6,919.42 of total excess payments). | The case shows an unusually high number of agency contact attempts (5 attempts)."
1,2,C-30644,0.957167,Suspended,high_award_excess | high_contact_attempts,"Actual payment amounts were substantially higher than the standard monthly award (maximum payment was 2.31x the award, resulting in $5,800.18 of total excess payments). | The case shows an unusually high number of agency contact attempts (6 attempts)."
2,3,C-33980,0.917495,Active,multi_payment_month | high_contact_attempts,Multiple payments were recorded within the same calendar month (case received 10 payments over 6 months). | The case shows an unusually high number of agency contact attempts (6 attempts).
3,4,C-30419,0.910496,Closed,high_standard_award | short_payment_history,"The case standard monthly award is unusually high ($1,760.46). | The case is closed and had a very short active payment history (only 2 transactions recorded)."
4,5,C-31298,0.882768,Active,high_award_excess | high_contact_attempts,"Actual payment amounts were substantially higher than the standard monthly award (maximum payment was 2.96x the award, resulting in $7,504.56 of total excess payments). | The case shows an unusually high number of agency contact attempts (5 attempts)."
5,6,C-30945,0.881573,Closed,post_closure_payment | high_contact_attempts,Payments continued after the recorded case closure (case received 3 post-closure payments). | The case shows an unusually high number of agency contact attempts (6 attempts).
6,7,C-30824,0.871548,Closed,post_closure_payment,Payments continued after the recorded case closure (case received 3 post-closure payments).
7,8,C-34118,0.863800,Active,high_award_excess | high_contact_attempts,"Actual payment amounts were substantially higher than the standard monthly award (maximum payment was 1.81x the award, resulting in $5,391.81 of total excess payments). | The case shows an unusually high number of agency contact attempts (5 attempts)."
8,9,C-32962,0.841570,Suspended,high_award_excess,"Actual payment amounts were substantially higher than the standard monthly award (maximum payment was 1.97x the award, resulting in $4,937.09 of total excess payments)."
9,10,C-32035,0.837788,Active,multi_payment_month | high_award_excess | high_contact_attempts | unreviewed_review_gap,"Multiple payments were recorded within the same calendar month (case received 7 payments over 6 months). | Actual payment amounts were substantially higher than the standard monthly award (maximum payment was 1.06x the award, resulting in $260.23 of total excess payments). | The case shows an unusually high number of agency contact attempts (5 attempts). | The case has been unreviewed for an unusually long period (19 months)."


## 5. Detailed Case-by-Case Explanation Inspection

We print a detailed view of each Top 20 case to verify that the generated explanation directly matches the actual underlying feature values.

In [5]:
for idx, row in df_top20.iterrows():
    print(f"Rank {row['rank']:02d} | Case: {row['case_id']} | Score: {row['priority_score']:.4f} | Status: {row['status']}")
    print(f"  - Monthly Award: ${row['monthly_award']:.2f}")
    print(f"  - Total Payments: {int(row['total_payments'])} (Months: {int(row['total_payments']) - int(row['post_closure_payment_count'])})")
    print(f"  - Post-Closure Payments: {int(row['post_closure_payment_count'])}")
    print(f"  - Total Excess Amount: ${row['total_excess_amount']:,.2f}")
    print(f"  - Multi-Payment in Month: {bool(row['has_same_month_multi_payments'])}")
    print(f"  - Contact Attempts: {int(row['contact_attempts'])} | Months Since Review: {int(row['months_since_review'])}")
    print(f"  - Signals: {row['top_contributing_signals']}")
    print(f"  - EXPLANATION: {row['plain_language_explanation']}")
    print('=' * 100 + '\n')

Rank 01 | Case: C-33263 | Score: 1.0000 | Status: Suspended
  - Monthly Award: $1368.13
  - Total Payments: 6 (Months: 6)
  - Post-Closure Payments: 0
  - Total Excess Amount: $6,919.42
  - Multi-Payment in Month: False
  - Contact Attempts: 5 | Months Since Review: 6
  - Signals: high_award_excess | high_contact_attempts
  - EXPLANATION: Actual payment amounts were substantially higher than the standard monthly award (maximum payment was 1.99x the award, resulting in $6,919.42 of total excess payments). | The case shows an unusually high number of agency contact attempts (5 attempts).

Rank 02 | Case: C-30644 | Score: 0.9572 | Status: Suspended
  - Monthly Award: $902.73
  - Total Payments: 6 (Months: 6)
  - Post-Closure Payments: 0
  - Total Excess Amount: $5,800.18
  - Multi-Payment in Month: False
  - Contact Attempts: 6 | Months Since Review: 13
  - Signals: high_award_excess | high_contact_attempts
  - EXPLANATION: Actual payment amounts were substantially higher than the standar

## 6. Step 7: Explainability Validation Checks

We run automated checks to guarantee our explanations comply with our quality constraints:
1. Exactly 20 cases are in the Top 20.
2. No Top 20 explanation is empty or null.
3. No demographic fields (e.g. `district`, `age_band`, `language_preference`, `tenure`) are present in explanations or keys.
4. No explanation claims fraud, guilt, or definitive improper payment (checks for illegal words like 'fraud', 'guilt', 'stole', 'illegal').

In [6]:
print('--- EXPLAINABILITY VALIDATION ---')
print(f'1. Count of Top 20 is exactly 20: {len(df_top20) == 20}')
print(f'2. Missing or empty explanations count: {df_top20["plain_language_explanation"].isnull().sum() + (df_top20["plain_language_explanation"] == "").sum()}')

# Demographics check
demographics_used = False
demographic_keywords = ['district', 'age_band', 'language', 'preference', 'tenure', 
                        'Calder', 'Weybridge', 'Northgate', 'Ash Hill', 'tenancy', 
                        'fixed abode', 'occupier', 'English', 'Spanish', 'Other']
for text in df_top20['plain_language_explanation'].dropna():
    if any(kw.lower() in text.lower() for kw in demographic_keywords):
        demographics_used = True
        print('Warning! Potential demographic reference found in:', text)
print(f'3. Demographic fields avoided: {not demographics_used}')

# Legality / Neutrally check
claims_guilt = False
guilt_keywords = ['fraud', 'guilt', 'cheat', 'illegal', 'stole', 'improper', 'wrongful']
for text in df_top20['plain_language_explanation'].dropna():
    if any(kw.lower() in text.lower() for kw in guilt_keywords):
        claims_guilt = True
        print('Warning! Guilt claim found in:', text)
print(f'4. Avoided claims of fraud/guilt/impropriety: {not claims_guilt}')

--- EXPLAINABILITY VALIDATION ---
1. Count of Top 20 is exactly 20: True
2. Missing or empty explanations count: 0
3. Demographic fields avoided: True
4. Avoided claims of fraud/guilt/impropriety: True


## 7. Save Explanation Dataset

We save the final Top 20 scored cases and explanations to the processed data directory.

In [7]:
output_path = '../data/processed/top20_explanations.csv'
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_top20.to_csv(output_path, index=False)
print(f'Top-20 explanations saved to {output_path}!')

Top-20 explanations saved to ../data/processed/top20_explanations.csv!


## Explainability Summary

### Explanation Method Selected
We implemented a **rule-based conditional explainer** that directly maps the engineered features associated with each prioritized case into a concise, human-readable sentence. This is a form of **post-hoc feature mapping**.

### Why This Approach is Appropriate
1. **Full Transparency & Reproducibility**: Unlike complex model-agnostic explanation approximations (like SHAP or LIME) which can be unstable, computationally expensive, and approximate, our method is completely deterministic, computationally light, and consistent with the exact metrics used to train the Isolation Forest.
2. **Aesthetic Suitability for Hackathon / Judges**: Plain-language descriptions with precise numerical metrics (e.g. exact post-closure payment counts, excess amounts, and contact attempt counts) provide direct, actionable evidence that is easy for a human auditor to review.
3. **Demographic Exclusivity**: By explicitly restricting the input fields available to the explainer, we guarantee that demographic attributes never leak into case descriptions.

### Main Types of Signals in the Top 20
1. **High Positive Discrepancies**: Cases receiving payments much larger than their standard monthly award (e.g. C-33263 and C-30644 getting over $5,800 in excess benefit amounts).
2. **Post-Closure Payment Leakage**: Closed cases continuing to receive payments (e.g. C-30945 and C-30824 receiving 3 payments after their declared closure month).
3. **Multi-Payment Month Disbursements**: Active cases receiving duplicate disbursements in a single month (e.g. C-33980 getting 10 payments over 6 months).
4. **Administrative Risk Factors**: Concurrently flagging high review gaps (e.g. unreviewed for > 18 months) and extreme contact frequencies (e.g. >= 5 attempts).

### Limitations of the Explanation Method
* **Threshold Dependencies**: The explainer relies on fixed heuristics (e.g. ratio > 1.05, contact >= 5) to flag features. While these capture the primary drivers of anomaly scores, they do not trace the non-linear interaction paths inside the Isolation Forest trees.
* **Observational Constraints**: Explanations only describe the historical transactions recorded in our window (July - December 2025) and cannot account for context outside this scope.